# CITYKIN WO1 — validation notebook

Validates the migration of the WH Cities "Similar (env)" dropdown to the WO6b-validated raw-curve
distance, plus the new terrain lens, **before any UI wiring** (WO1's own validation order).

**Steps run here:**
1. Assemble the substrate: 254 basin-joined WH Cities x (raw monthly curves, `ari_log` aridity,
   point-window terrain facets from `gaz.wh_cities_terrain`).
2. Per-lens correlation check on the terrain lens's three facets (Part B proviso: metric-within-lens
   is decided by a correlation check on the 254, not assumed).
3. Tbilisi's own point-window terrain, fetched fresh (it is **not** a `gaz.wh_cities` row — confirmed;
   see the tracker's Locked decisions). Ranking + the Part D fixture verdict come in a later cell,
   added once the correlation check (step 2) settles which metric the terrain lens uses.

WO: `docs/cdop/citykin/wo1_update-whcities.md`. Tracker: `docs/cdop/citykin/CITYKIN_tracker.md`.

In [1]:
# Cell 1
%matplotlib inline
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from scripts.shared.db_utils import db_connect
import scripts.shared.db_utils as db_utils

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop' / 'citykin'
OUT.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}\nOUT : {OUT}")

ROOT: /Users/karlg/Documents/repos/_edops
OUT : /Users/karlg/Documents/repos/_edops/output/cdop/citykin


In [2]:
# Cell 2 -- assemble the WH Cities substrate (raw monthly curves + ari_log + terrain facets).
# gaz.wh_cities.basin_id references basin08.id (a serial PK), NOT basin08.hybas_id directly --
# confirmed by inspection before writing this query (basin08.id is a small integer; hybas_id is the
# real ~10-digit HydroBASINS code the persist view is keyed on). -9999 masked before any use (CLAUDE.md).
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()

cities = pd.read_sql("""
    SELECT c.id AS city_id, c.city, c.country, c.region,
           ST_Y(c.geom) AS lat, ST_X(c.geom) AS lon,
           b.hybas_id, b.ari_ix_sav,
           t.grid_elev_mean, t.relief_range_m, t.landform_position
    FROM gaz.wh_cities c
    JOIN basin08 b ON b.id = c.basin_id
    JOIN gaz.wh_cities_terrain t ON t.city_id = c.id
    WHERE c.basin_id IS NOT NULL
    ORDER BY c.id
""", conn)
cities['hybas_id'] = cities['hybas_id'].astype('int64')
cities['ari_ix_sav'] = cities['ari_ix_sav'].replace(-9999, np.nan)

hy = tuple(int(x) for x in sorted(cities['hybas_id'].unique()))
arr = pd.read_sql(f"""
    SELECT hybas_id, pre_mm_monthly, tmp_dc_monthly
    FROM public.v_basin08_persist_rev2
    WHERE hybas_id IN {hy}
""", conn)
conn.close()
arr['hybas_id'] = arr['hybas_id'].astype('int64')

PRE = np.array(arr['pre_mm_monthly'].tolist(), dtype=float)
TMP = np.array(arr['tmp_dc_monthly'].tolist(), dtype=float)   # already deg C, not x10
der = pd.DataFrame({
    'hybas_id':           arr['hybas_id'].to_numpy(),
    'temperature_annual': TMP.mean(axis=1),
    'tmp_seas_amp':       TMP.max(axis=1) - TMP.min(axis=1),
    'pre_mm_monthly':     [r.tolist() for r in PRE],
    'tmp_dc_monthly':     [r.tolist() for r in TMP],
})

sub = cities.merge(der, on='hybas_id', how='left')
sub['ari_log'] = np.log1p(sub['ari_ix_sav'])   # carried from WO8b/8c drop-to-representative metric

out_path = OUT / 'wo1_substrate.parquet'
sub.to_parquet(out_path, index=False)

lines = [
    f"substrate rows: {len(sub)}  |  unique basins: {sub['hybas_id'].nunique()}  "
    f"(dup basins mean two cities share an L08 basin -- expected, not an error)",
    f"saved -> {out_path}",
    "",
    "NaN counts per column (coverage check):",
    sub[['ari_ix_sav', 'ari_log', 'temperature_annual', 'tmp_seas_amp',
         'grid_elev_mean', 'relief_range_m', 'landform_position']].isna().sum().to_string(),
]
print("\n".join(lines))

substrate rows: 254  |  unique basins: 241  (dup basins mean two cities share an L08 basin -- expected, not an error)
saved -> /Users/karlg/Documents/repos/_edops/output/cdop/citykin/wo1_substrate.parquet

NaN counts per column (coverage check):
ari_ix_sav            0
ari_log               0
temperature_annual    0
tmp_seas_amp          0
grid_elev_mean        0
relief_range_m        0
landform_position     0


In [3]:
# Cell 3 -- Part B proviso: correlation check on the terrain lens's three candidate facets, on the
# corpus, not assumed. |r| >= 0.70 is the same Mahalanobis-or-drop threshold used throughout WO8.
cols = ['grid_elev_mean', 'relief_range_m', 'landform_position']
ok = sub.dropna(subset=cols).reset_index(drop=True)
corr = ok[cols].corr()

hi = [(cols[i], cols[j], round(corr.iat[i, j], 2))
      for i in range(len(cols)) for j in range(i + 1, len(cols)) if abs(corr.iat[i, j]) >= 0.70]

print("\n".join([
    f"terrain facet correlation (n={len(ok)} complete-case):",
    corr.round(2).to_string(),
    "",
    f"|r| >= 0.70 pairs (Mahalanobis / drop candidates): {hi if hi else 'none'}",
]))

terrain facet correlation (n=254 complete-case):
                   grid_elev_mean  relief_range_m  landform_position
grid_elev_mean               1.00            0.43              -0.05
relief_range_m               0.43            1.00               0.01
landform_position           -0.05            0.01               1.00

|r| >= 0.70 pairs (Mahalanobis / drop candidates): none


In [4]:
# Cell 4 -- Tbilisi (41.6938, 44.8015; canonical project coordinates, used consistently since WO5) is
# not a gaz.wh_cities row (confirmed; tracker Locked decisions) -- fetch its own point-window terrain
# grid the same way persist_whcities_terrain.py does for the corpus, so it can be ranked against it.
# Radius matches the corpus script: +-10km/5km-spacing, widened from the WO8c-inherited +-2km after
# Cells 7-8 showed 2km was too small to reach the enclosing highlands of the WO's own named "high
# valley floor" fixture candidates, and that 10km/25pts agrees closely with a much denser 81pt grid.
import json as _json
import math
import ssl
from urllib.parse import urlencode
from urllib.request import Request, urlopen

try:
    import certifi
except ImportError:
    certifi = None

GRID_STEPS_KM = [-10, -5, 0, 5, 10]
KM_PER_DEG_LAT = 111.32
TBILISI_LAT, TBILISI_LON = 41.6938, 44.8015

def _grid_points(lat, lon):
    dlat = 1.0 / KM_PER_DEG_LAT
    dlon = 1.0 / (KM_PER_DEG_LAT * max(math.cos(math.radians(lat)), 1e-6))
    return [(lat + dy * dlat, lon + dx * dlon) for dy in GRID_STEPS_KM for dx in GRID_STEPS_KM]

def _fetch_elevations(points):
    locs = "|".join(f"{lat},{lon}" for lat, lon in points)
    url = f"https://api.opentopodata.org/v1/mapzen?{urlencode({'locations': locs})}"
    req = Request(url, headers={"Accept": "application/json", "User-Agent": "edop-cdop/0.1"})
    ctx = ssl.create_default_context(cafile=certifi.where()) if certifi else ssl.create_default_context()
    with urlopen(req, timeout=15.0, context=ctx) as resp:
        payload = _json.loads(resp.read().decode("utf-8"))
    if payload.get("status") != "OK":
        return [None] * len(points)
    results = payload.get("results") or []
    return [r.get("elevation") for r in results] if len(results) == len(points) else [None] * len(points)

pts = _grid_points(TBILISI_LAT, TBILISI_LON)
vals = _fetch_elevations(pts)   # single batch, 25 points, well under the 100/request cap
resolved = [v for v in vals if v is not None]
vmin, vmax, vmean = min(resolved), max(resolved), sum(resolved) / len(resolved)
tbilisi_terrain = {
    'grid_elev_min': vmin, 'grid_elev_max': vmax, 'grid_elev_mean': vmean,
    'relief_range_m': vmax - vmin,
    'landform_position': (vmean - vmin) / (vmax - vmin) if vmax > vmin else None,
}

corpus_pct = {k: float((sub[k] < v).mean() * 100) for k, v in tbilisi_terrain.items() if k in sub.columns}

print("\n".join([
    f"Tbilisi point-window terrain ({len(resolved)}/{len(pts)} grid points resolved, +-10km box):",
    _json.dumps(tbilisi_terrain, indent=2),
    "",
    "Tbilisi's percentile within the 254-city corpus (per facet):",
    _json.dumps({k: round(v, 1) for k, v in corpus_pct.items()}, indent=2),
]))

Tbilisi point-window terrain (25/25 grid points resolved, +-10km box):
{
  "grid_elev_min": 366.0,
  "grid_elev_max": 1115.0,
  "grid_elev_mean": 673.0,
  "relief_range_m": 749.0,
  "landform_position": 0.4098798397863818
}

Tbilisi's percentile within the 254-city corpus (per facet):
{
  "grid_elev_mean": 79.9,
  "relief_range_m": 81.9,
  "landform_position": 55.5
}


In [5]:
# Cell 5 -- terrain lens metric decision (Part B proviso): no facet pair clears the |r|>=0.70
# Mahalanobis/drop threshold (max |r|=0.35, elevation vs relief_range) -- the three facets are near-
# independent on this corpus. Decision: plain Euclidean on z-scored (elevation, relief_range, landform
# position); no Mahalanobis, no drop-to-representative. Same lens discipline as WO8, applied here.
TERRAIN_COLS = ['grid_elev_mean', 'relief_range_m', 'landform_position']

def terrain_backdrop_z(df):
    ok = df.dropna(subset=TERRAIN_COLS).reset_index(drop=True)
    X = ok[TERRAIN_COLS].to_numpy(float)
    mu, sd = X.mean(0), X.std(0)
    return ok, (X - mu) / sd, mu, sd

backdrop, Xz, terrain_mu, terrain_sd = terrain_backdrop_z(sub)
print(f"terrain backdrop: n={len(backdrop)} (all {len(sub)} cities have complete terrain facets) "
      f"| metric: Euclidean on z-scored {TERRAIN_COLS}")

terrain backdrop: n=254 (all 254 cities have complete terrain facets) | metric: Euclidean on z-scored ['grid_elev_mean', 'relief_range_m', 'landform_position']


In [6]:
# Cell 6 -- Part D fixture: rank the corpus by Euclidean distance (z-scored terrain facets) to
# Tbilisi's own point-window terrain from Cell 4. Accept gate: top results should be high
# *valley-floor* cities (candidates named in the WO: Kathmandu, Quito, Mexico City, Bogota, Sanaa),
# not high-flat or high-peak cities that share only elevation. Needs Cell 4 run first (tbilisi_terrain).
tb_vec = np.array([tbilisi_terrain[c] for c in TERRAIN_COLS], dtype=float)
tb_z = (tb_vec - terrain_mu) / terrain_sd

dist = np.sqrt(((Xz - tb_z) ** 2).sum(axis=1))
ranked = backdrop.assign(dist_to_tbilisi=dist).sort_values('dist_to_tbilisi')

cols_show = ['city', 'country', 'dist_to_tbilisi', 'grid_elev_mean', 'relief_range_m', 'landform_position']
print("Tbilisi point-window terrain:", {c: round(tbilisi_terrain[c], 1) for c in TERRAIN_COLS})
print()
print("nearest 12 cities by terrain distance:")
print(ranked[cols_show].head(12).to_string(index=False))

Tbilisi point-window terrain: {'grid_elev_mean': 673.0, 'relief_range_m': 749.0, 'landform_position': 0.4}

nearest 12 cities by terrain distance:
             city     country  dist_to_tbilisi  grid_elev_mean  relief_range_m  landform_position
Palazzolo Acreide       Italy         0.368074          562.36           596.0           0.410000
           Padula       Italy         0.555583          792.08           944.0           0.366610
            Úbeda       Spain         0.619743          519.80           487.0           0.424641
            Baeza       Spain         0.638671          472.64           514.0           0.374786
         Bardejov    Slovakia         0.669343          479.48           790.0           0.327190
          Yerevan     Armenia         0.675977         1147.56           749.0           0.421308
          Yangsan South Korea         0.676899          247.48           630.0           0.386476
        Dubrovnik     Croatia         0.793908          114.84       

In [7]:
# Cell 7 -- radius probe: none of the WO's named "high valley floor" candidates (Kathmandu, Quito,
# Mexico City, Sanaa; Bogota is not in the corpus) land anywhere near the top of Cell 6's ranking --
# they rank 174th/248th/239th/243rd of 254. Their point-window relief is near-flat (17-73m) because
# the +-2km box (inherited unchanged from WO8c's society-terrain use case) is too small to reach the
# distant enclosing highlands that actually define a classic intermontane basin. Karl: "2km is not far
# at all". Probe wider boxes (5/10/20km, still a 5x5=25-point grid, just wider spacing) on Tbilisi +
# the 5 named candidates only -- NOT the full 254 -- to see at what radius the enclosure signal
# appears, before deciding whether/how to redo the full corpus.
import time

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()
probe_cities = pd.read_sql("""
    SELECT city, country, ST_Y(geom) AS lat, ST_X(geom) AS lon
    FROM gaz.wh_cities
    WHERE city IN ('Kathmandu', 'Mexico City', 'Sanaa', 'Quito', 'Cusco')
""", conn)
conn.close()
probe_pts = [('Tbilisi', 'Georgia', TBILISI_LAT, TBILISI_LON)] + list(probe_cities.itertuples(index=False, name=None))

RADII_KM = [2, 5, 10, 20]
BATCH_SIZE = 100

def _grid_points_r(lat, lon, radius_km, n=5):
    steps = np.linspace(-radius_km, radius_km, n)
    dlat_per_km = 1.0 / KM_PER_DEG_LAT
    dlon_per_km = 1.0 / (KM_PER_DEG_LAT * max(math.cos(math.radians(lat)), 1e-6))
    return [(lat + dy * dlat_per_km, lon + dx * dlon_per_km) for dy in steps for dx in steps]

queue = []   # (city, radius_km, lat, lon)
for city, country, lat, lon in probe_pts:
    for r in RADII_KM:
        queue.extend((city, r, plat, plon) for plat, plon in _grid_points_r(lat, lon, r))

elevs = {}
n_batches = (len(queue) - 1) // BATCH_SIZE + 1
for b in range(n_batches):
    chunk = queue[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    vals = _fetch_elevations([(lat, lon) for _, _, lat, lon in chunk])
    for (city, r, lat, lon), v in zip(chunk, vals):
        elevs.setdefault((city, r), []).append(v)
    time.sleep(1.05)
print(f"fetched {len(queue)} points in {n_batches} batches")

rows = []
for city, country, lat, lon in probe_pts:
    for r in RADII_KM:
        resolved = [v for v in elevs[(city, r)] if v is not None]
        vmin, vmax, vmean = min(resolved), max(resolved), sum(resolved) / len(resolved)
        relief = vmax - vmin
        landform = (vmean - vmin) / relief if relief > 0 else None
        rows.append({'city': city, 'radius_km': r, 'n_resolved': len(resolved),
                      'grid_elev_mean': round(vmean, 1), 'relief_range_m': round(relief, 1),
                      'landform_position': round(landform, 2) if landform is not None else None})

probe_df = pd.DataFrame(rows)
print(probe_df.pivot(index='city', columns='radius_km',
                      values=['relief_range_m', 'landform_position']).to_string())

fetched 600 points in 6 batches
            relief_range_m                         landform_position                  
radius_km               2       5       10      20                2     5     10    20
city                                                                                  
Cusco                416.0   740.0   890.0  1809.0              0.48  0.54  0.48  0.50
Kathmandu             73.0   260.0   987.0  2005.0              0.45  0.18  0.27  0.50
Mexico City           17.0    67.0   270.0  1414.0              0.29  0.19  0.16  0.15
Quito                600.0  1672.0  2397.0  1619.0              0.23  0.31  0.32  0.48
Sanaa                 37.0   510.0   521.0   764.0              0.44  0.21  0.35  0.43
Tbilisi              410.0   693.0   749.0  1062.0              0.31  0.38  0.41  0.40


In [8]:
# Cell 8 -- density probe: same 6 cities, same radii (10/20km) but a 9x9=81-point grid instead of
# 5x5=25, to check whether landform_position's Kathmandu/Quito wobble was real signal or just grid
# aliasing (25 points at 10km spacing can miss/catch a ridge close to arbitrarily). Still probe-only,
# not the full 254.
queue2 = []   # (city, radius_km, lat, lon)
for city, country, lat, lon in probe_pts:
    for r in [10, 20]:
        queue2.extend((city, r, plat, plon) for plat, plon in _grid_points_r(lat, lon, r, n=9))

elevs2 = {}
n_batches2 = (len(queue2) - 1) // BATCH_SIZE + 1
for b in range(n_batches2):
    chunk = queue2[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    vals = _fetch_elevations([(lat, lon) for _, _, lat, lon in chunk])
    for (city, r, lat, lon), v in zip(chunk, vals):
        elevs2.setdefault((city, r), []).append(v)
    time.sleep(1.05)
print(f"fetched {len(queue2)} points in {n_batches2} batches")

rows2 = []
for city, country, lat, lon in probe_pts:
    for r in [10, 20]:
        resolved = [v for v in elevs2[(city, r)] if v is not None]
        vmin, vmax, vmean = min(resolved), max(resolved), sum(resolved) / len(resolved)
        relief = vmax - vmin
        landform = (vmean - vmin) / relief if relief > 0 else None
        rows2.append({'city': city, 'radius_km': r, 'n_pts': len(resolved), 'grid': '9x9',
                       'relief_range_m': round(relief, 1),
                       'landform_position': round(landform, 2) if landform is not None else None})

dense_df = pd.DataFrame(rows2)
print()
print("9x9 grid (81 pts) vs the earlier 5x5 (25 pts), side by side:")
compare = dense_df.merge(
    probe_df[probe_df['radius_km'].isin([10, 20])][['city', 'radius_km', 'relief_range_m', 'landform_position']],
    on=['city', 'radius_km'], suffixes=('_9x9', '_5x5'))
print(compare[['city', 'radius_km', 'relief_range_m_5x5', 'relief_range_m_9x9',
                'landform_position_5x5', 'landform_position_9x9']].to_string(index=False))

fetched 972 points in 10 batches

9x9 grid (81 pts) vs the earlier 5x5 (25 pts), side by side:
       city  radius_km  relief_range_m_5x5  relief_range_m_9x9  landform_position_5x5  landform_position_9x9
    Tbilisi         10               749.0              1000.0                   0.41                   0.31
    Tbilisi         20              1062.0              1062.0                   0.40                   0.41
      Cusco         10               890.0              1099.0                   0.48                   0.47
      Cusco         20              1809.0              1809.0                   0.50                   0.49
  Kathmandu         10               987.0              1183.0                   0.27                   0.20
  Kathmandu         20              2005.0              2005.0                   0.50                   0.49
Mexico City         10               270.0               274.0                   0.16                   0.16
Mexico City         20           

In [9]:
# Cell 9 -- Karl's redesign: elevation is a "contained by elevation" ELIGIBILITY gate, not a
# continuous facet to compare magnitudes on -- a 500m valley floor and a 3000m valley floor can both
# be genuinely "high, contained" places; a raw z-scored elevation term was instead measuring how far
# apart their absolute heights are, which is why Kathmandu/Cusco/Quito/Sanaa/Mexico City (all
# 1100-3800m) never ranked near Tbilisi (673m) in Cell 6 despite plausible containment shape.
#
# Threshold: grid_elev_mean >= 400m. Not arbitrary -- the corpus histogram (run separately, confirmed
# against gaz.wh_cities/gaz.wh_cities_terrain, not the WHG gazetteer) has a genuine empty bin at
# 350-375m (0 cities) between a thin tail below (1 city, 325-350m) and a thin tail above (2 cities,
# 375-400m: Notodden, Caceres) -- the same "real histogram trough" shape as the WO2 aridity gate, not
# a fitted line. 400m sits just above the trough, clear of the edge cities, comfortably below Tbilisi
# (673m) and every named fixture candidate (1147-3756m).
#
# Ranking metric for gated-in cities: Euclidean on z-scored (relief_range_m, landform_position) only
# -- elevation drops out of the DISTANCE once eligibility is established, per Karl's steer.
ELEV_HIGH_THRESHOLD = 400.0
SHAPE_COLS = ['relief_range_m', 'landform_position']

eligible = sub[sub['grid_elev_mean'] >= ELEV_HIGH_THRESHOLD].reset_index(drop=True)
X2 = eligible[SHAPE_COLS].to_numpy(float)
shape_mu, shape_sd = X2.mean(0), X2.std(0)
Xz2 = (X2 - shape_mu) / shape_sd

print(f"gate: grid_elev_mean >= {ELEV_HIGH_THRESHOLD:.0f}m  ->  {len(eligible)} / {len(sub)} cities eligible")
print(f"Tbilisi (673.0m) clears the gate: {673.0 >= ELEV_HIGH_THRESHOLD}")
print(f"ranking metric: Euclidean on z-scored {SHAPE_COLS}, fit on the {len(eligible)}-city eligible pool")

gate: grid_elev_mean >= 400m  ->  83 / 254 cities eligible
Tbilisi (673.0m) clears the gate: True
ranking metric: Euclidean on z-scored ['relief_range_m', 'landform_position'], fit on the 83-city eligible pool


In [10]:
# Cell 10 -- Part D fixture, rerun under the gate + 2-facet (relief, position) distance.
tb_vec2 = np.array([tbilisi_terrain[c] for c in SHAPE_COLS], dtype=float)
tb_z2 = (tb_vec2 - shape_mu) / shape_sd

dist2 = np.sqrt(((Xz2 - tb_z2) ** 2).sum(axis=1))
ranked2 = eligible.assign(dist_to_tbilisi=dist2).sort_values('dist_to_tbilisi').reset_index(drop=True)
ranked2['rank'] = ranked2.index + 1

cols_show = ['rank', 'city', 'country', 'dist_to_tbilisi', 'grid_elev_mean', 'relief_range_m', 'landform_position']
print("nearest 12 cities (gated, 2-facet distance):")
print(ranked2[cols_show].head(12).to_string(index=False))

print()
print("named fixture candidates' new rank (of", len(ranked2), "eligible):")
for name in ['Kathmandu', 'Mexico City', 'Sanaa', 'Quito', 'Cusco', 'Yerevan', 'Shibam']:
    row = ranked2[ranked2['city'] == name]
    if len(row):
        print(row[cols_show].to_string(index=False, header=False))
    else:
        print(f"{name}: not in the gated pool (below {ELEV_HIGH_THRESHOLD:.0f}m)")

nearest 12 cities (gated, 2-facet distance):
 rank              city country  dist_to_tbilisi  grid_elev_mean  relief_range_m  landform_position
    1           Yerevan Armenia         0.096011         1147.56           749.0           0.421308
    2 Palazzolo Acreide   Italy         0.305659          562.36           596.0           0.410000
    3           Lijiang   China         0.339566         2737.36           912.0           0.398421
    4         Bhaktapur   Nepal         0.365200         1539.44           634.0           0.376088
    5        Guanajuato  Mexico         0.466047         2246.60           828.0           0.462077
    6            Padula   Italy         0.532821          792.08           944.0           0.366610
    7             Úbeda   Spain         0.537902          519.80           487.0           0.424641
    8             Baeza   Spain         0.554370          472.64           514.0           0.374786
    9             Sucre Bolivia         0.602925       

## WO1a — correcting the terrain lens: query-relative tolerance knobs, not a global gate

Cells 1-10 above are WO1's original design and its validation record — kept as history, **superseded**
by what follows. WO1's `grid_elev_mean >= 400m` eligibility gate passed the Tbilisi fixture but turned
out to be a Tbilisi-specific artifact: it hard-codes "high" as a global constant, so a flat query city
(Bruges) would be excluded by its own gate, and the shape terms are meaningless for a delta city.
Passing-for-Tbilisi and generalizing-to-all-254-cities turned out to be different things.

**The fix** (`docs/cdop/citykin/wo1a_terrain-lens.md`): retire the gate; rebuild as three
**query-relative tolerance knobs** (elevation, relief, landform-position), each anchored to the
*selected city's own* facet values — the pattern already validated for the sandbox's climate-regime
lenses. Factored core: `scripts/cdop/citykin/terrain_lens.py` (`rank_by_terrain`), separate from any
presentation head, so a later WO can wire the same core to a paint-a-set head on the sandbox
Similarity tab without reimplementing the distance.

**Also fixed along the way**: a real data-quality issue in `gaz.wh_cities_terrain` — OpenTopoData's
`mapzen` dataset returns bathymetric depths (not null) for grid points that fall in open water, which
contaminated 88/254 cities' relief statistics (worst case: Willemstad's grid spanned -1249m to 67m).
Fixed in `scripts/cdop/citykin/terrain_grid.py` (shared fetch+compute module, now the single
implementation used by the persist script, this notebook, and eventually a live query-by-coordinate
API path) by dropping grid points with elevation < 0 before computing relief stats. 253/254 cities now
resolve (Aktau, Kazakhstan is the one casualty — genuinely mostly Caspian Sea within its own ±10km box,
confirmed by satellite view, not a bug).

Two fixtures now (WO1a Part D) — Tbilisi (contained high valley) and Bruges (flat, near-sea-level,
the generalization check WO1's single fixture missed) — both must pass at the *same* default knob
settings, no per-city tuning.

In [11]:
# Cell 11 -- reload the corpus terrain facets post negative-elevation-filter fix (persist script
# rerun already), and re-check the facet correlation now that bathymetric contamination is gone.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()
terrain_corpus = pd.read_sql("""
    SELECT c.id AS city_id, c.city, c.country, t.grid_elev_mean, t.relief_range_m,
           t.landform_position, t.n_grid_land, t.n_grid_points
    FROM gaz.wh_cities c JOIN gaz.wh_cities_terrain t ON t.city_id = c.id
    WHERE c.basin_id IS NOT NULL
""", conn)
conn.close()

TERRAIN_FACETS = ['grid_elev_mean', 'relief_range_m', 'landform_position']
resolved_mask = terrain_corpus[TERRAIN_FACETS].notna().all(axis=1)
corr = terrain_corpus.loc[resolved_mask, TERRAIN_FACETS].corr()

print("\n".join([
    f"terrain corpus: {resolved_mask.sum()} / {len(terrain_corpus)} resolved "
    f"(Aktau is the one non-resolving case -- see markdown above)",
    "",
    "facet correlation, post land-filter fix:",
    corr.round(2).to_string(),
    "(elevation-vs-relief rose from 0.43 pre-fix to 0.62 here -- still under the 0.70 bar, noted)",
]))

terrain corpus: 253 / 254 resolved (Aktau is the one non-resolving case -- see markdown above)

facet correlation, post land-filter fix:
                   grid_elev_mean  relief_range_m  landform_position
grid_elev_mean               1.00            0.62               0.08
relief_range_m               0.62            1.00              -0.17
landform_position            0.08           -0.17               1.00
(elevation-vs-relief rose from 0.43 pre-fix to 0.62 here -- still under the 0.70 bar, noted)


In [12]:
# Cell 12 -- the factored tolerance core (scripts/cdop/citykin/terrain_lens.py, imported not
# reimplemented -- the module the retrieval head and, later, a sandbox paint-a-set head will both
# call). Fetch Tbilisi fresh via the shared terrain_grid module (same one the persist script uses),
# confirming no water contamination (Tbilisi is inland/mountainous).
from scripts.cdop.citykin.terrain_grid import point_window_terrain
from scripts.cdop.citykin.terrain_lens import rank_by_terrain, FACETS

tbilisi_terrain2 = point_window_terrain(41.6938, 44.8015)

print("\n".join([
    "Tbilisi (fresh fetch via terrain_grid.point_window_terrain):",
    str({k: round(tbilisi_terrain2[k], 2) for k in
         ['grid_elev_mean', 'relief_range_m', 'landform_position', 'n_grid_land', 'n_grid_points']}),
]))

Tbilisi (fresh fetch via terrain_grid.point_window_terrain):
{'grid_elev_mean': 673.0, 'relief_range_m': 749.0, 'landform_position': 0.41, 'n_grid_land': 25, 'n_grid_points': 25}


In [13]:
# Cell 13 -- Part D fixture 1: Tbilisi. Locked joint defaults (set against both fixtures together,
# not either alone -- WO1a Part D): elevation +-500m, relief +-300m, landform-position +-0.10,
# elev_weight=1.0 (ranking distance normalizes each facet's deviation by its OWN tolerance, so
# elevation informs the ranking without a corpus-wide z-score dominating it -- WO1's original failure).
LOCKED_TOLERANCES = {'grid_elev_mean': 500.0, 'relief_range_m': 300.0, 'landform_position': 0.10}
LOCKED_ELEV_WEIGHT = 1.0

corpus_resolved = terrain_corpus[resolved_mask].reset_index(drop=True)

ranked_tbilisi = rank_by_terrain(tbilisi_terrain2, corpus_resolved, LOCKED_TOLERANCES, LOCKED_ELEV_WEIGHT)
cols_show = ['city', 'country', 'terrain_dist'] + FACETS
print(f"Tbilisi @ locked defaults {LOCKED_TOLERANCES}: {len(ranked_tbilisi)} eligible of {len(corpus_resolved)}")
print(ranked_tbilisi[cols_show].head(12).to_string(index=False))

Tbilisi @ locked defaults {'grid_elev_mean': 500.0, 'relief_range_m': 300.0, 'landform_position': 0.1}: 22 eligible of 253
             city     country  terrain_dist  grid_elev_mean  relief_range_m  landform_position
Palazzolo Acreide       Italy      0.555937          562.36           596.0           0.410000
           Padula       Italy      0.816362          792.08           944.0           0.366610
         Bardejov    Slovakia      0.923169          479.48           790.0           0.327190
            Úbeda       Spain      0.937219          519.80           487.0           0.424641
            Baeza       Spain      0.947283          472.64           514.0           0.374786
          Yerevan     Armenia      0.955976         1147.56           749.0           0.421308
          Yangsan South Korea      0.967671          247.48           630.0           0.386476
          Derbent      Russia      1.049513          242.50           582.0           0.387457
       Ouro Preto     

In [14]:
# Cell 14 -- Part D fixture 2: Bruges -- the generalization check WO1's single (Tbilisi-only) fixture
# missed. Bruges IS a gaz.wh_cities row (unlike Tbilisi), so its query facets come straight from the
# corpus; excluded from its own result set. Same locked defaults as Cell 13, no per-city tuning.
bruges_idx = corpus_resolved.index[corpus_resolved['city'] == 'Bruges'][0]
bruges_query = {f: corpus_resolved.loc[bruges_idx, f] for f in FACETS}

ranked_bruges = rank_by_terrain(bruges_query, corpus_resolved, LOCKED_TOLERANCES, LOCKED_ELEV_WEIGHT,
                                 exclude_index=bruges_idx)
print(f"Bruges @ locked defaults {LOCKED_TOLERANCES}: {len(ranked_bruges)} eligible of {len(corpus_resolved)}")
print(ranked_bruges[cols_show].head(12).to_string(index=False))

Bruges @ locked defaults {'grid_elev_mean': 500.0, 'relief_range_m': 300.0, 'landform_position': 0.1}: 35 eligible of 253
                city       country  terrain_dist  grid_elev_mean  relief_range_m  landform_position
Island of Mozambique    Mozambique      0.260456       11.000000            37.0           0.243243
         Tlacotalpan        Mexico      0.401569        8.120000            40.0           0.178000
              Bolgar        Russia      0.404839       74.880000           136.0           0.219706
                 Huế       Vietnam      0.419712       15.956522            58.0           0.257871
Santa Cruz de Mompox      Colombia      0.419730       19.880000            42.0           0.259048
           Singapore     Singapore      0.494425       18.222222            65.0           0.264957
              Lübeck       Germany      0.546921       21.680000            73.0           0.269589
               Tunis       Tunisia      0.552255       42.500000           185